# Real-World Dataset Analysis Project
## Breast Cancer Wisconsin (Diagnostic) Dataset
**Data Analyst Internship — Task 3**

---

## Project report description

This project presents an end-to-end exploratory analysis of the Breast Cancer Wisconsin (Diagnostic) dataset, a public research dataset containing measurements computed from digitised images of breast-mass cell nuclei. The dataset contains 569 observations, 30 numerical predictor variables and one diagnosis target. Each observation represents a breast mass, while the predictors describe characteristics such as radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry and fractal dimension. The purpose of the analysis is to understand the data, assess its quality, discover which measurements differ most between benign and malignant cases, explore relationships among variables, and evaluate whether the two diagnosis groups can be separated using a simple baseline machine-learning model.

The notebook follows a reproducible workflow using Python, Pandas, NumPy, SciPy, Matplotlib, Seaborn and scikit-learn. It begins by loading the dataset and examining its shape, column types, summary statistics and target distribution. Data-quality checks are then performed for missing values, duplicate rows, negative measurements and unusual zero values. Since the dataset is already clean, the preprocessing stage focuses on converting the target into readable diagnosis labels, standardising column names and screening for outliers with the interquartile-range method. Outliers are retained because unusually large or irregular measurements may represent meaningful observations rather than data-entry mistakes.

Exploratory analysis uses descriptive statistics, histograms, box plots, correlation analysis and principal component analysis. The analysis compares the distributions of benign and malignant cases and ranks features using Cohen's d and Mann–Whitney U tests with a Bonferroni adjustment. Finally, a standardised logistic-regression pipeline is evaluated with stratified five-fold cross-validation to reduce data leakage. The findings are interpreted in the context of the dataset, with limitations clearly stated. This is an educational statistical analysis only; it is not a clinical diagnostic system and must not be used to make medical decisions.

## 1. Objectives

1. Select and understand a reliable real-world dataset.
2. Explore its structure, data types, distributions and data quality.
3. Clean and preprocess the data where necessary.
4. Perform exploratory data analysis using Pandas, NumPy and statistical methods.
5. Create visualisations that communicate important patterns.
6. Summarise findings, limitations and practical recommendations.

## 2. Dataset source and context

The dataset was created by William H. Wolberg, W. Nick Street and Olvi L. Mangasarian at the University of Wisconsin. It is distributed through the UCI Machine Learning Repository and is also bundled with scikit-learn. Each row describes one breast mass using 30 measurements derived from cell nuclei. For each of ten characteristics, the dataset records the mean, standard error and worst value.

**Important target mapping in scikit-learn:** `0 = malignant`, `1 = benign`. The mapping is taken from the dataset metadata rather than assumed.

In [ ]:
# Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from IPython.display import display

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 3. Load the dataset

In [ ]:
# Load the built-in copy of the Wisconsin Diagnostic dataset
raw = load_breast_cancer()
df = pd.DataFrame(raw.data, columns=raw.feature_names)
df["target"] = raw.target

label_map = {i: str(name).capitalize() for i, name in enumerate(raw.target_names)}
data = df.copy()
data.columns = [c.replace(" ", "_") for c in data.columns]
data["diagnosis"] = pd.Categorical(
    data["target"].map(label_map),
    categories=["Benign", "Malignant"]
)
features = [c for c in data.columns if c not in ("target", "diagnosis")]

print("Dataset shape:", data.shape)
print("Target mapping:", label_map)
print("Number of predictor features:", len(features))
display(data.head())

## 4. Data exploration

In [ ]:
print("Data types:")
display(data.dtypes.value_counts().rename("number_of_columns").to_frame())

print("Dataset information:")
data.info()

print("Descriptive statistics:")
display(data[features].describe().T)

print("Diagnosis counts:")
class_counts = data["diagnosis"].value_counts().reindex(["Benign", "Malignant"])
class_summary = pd.DataFrame({
    "count": class_counts,
    "percent": 100 * class_counts / len(data)
})
display(class_summary)

In [ ]:
# Visualise class balance
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="diagnosis", order=["Benign", "Malignant"])
plt.title("Number of Breast Masses by Diagnosis")
plt.xlabel("Diagnosis")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

## 5. Data-quality checks

In [ ]:
missing_total = int(data.isna().sum().sum())
duplicate_rows = int(data.duplicated().sum())
negative_values = int((data[features] < 0).sum().sum())
target_values = sorted(data["target"].unique().tolist())

print("Total missing values:", missing_total)
print("Duplicate rows:", duplicate_rows)
print("Negative feature values:", negative_values)
print("Target values present:", target_values)

zero_counts = (data[features] == 0).sum()
print("\nFeatures containing zero values:")
display(zero_counts[zero_counts > 0].to_frame("zero_count"))

In [ ]:
# Internal consistency check: compare area with pi * radius^2
ratio = data["mean_area"] / (np.pi * data["mean_radius"] ** 2)
print("Area / (pi * radius squared) summary:")
display(ratio.describe().to_frame("ratio"))

**Interpretation:** A clean dataset does not require artificial changes. The expected checks should show no missing values, duplicate rows or negative measurements. Zero values in concavity-related variables can be valid: they indicate that no concave portions were measured. The area-to-radius comparison is a rough consistency check, not an identity, because the measured boundary need not be a perfect circle.

## 6. Cleaning and preprocessing

In [ ]:
# The original target is retained for modelling.
# Create readable categorical labels and use consistent snake_case columns.
print("Diagnosis data type:", data["diagnosis"].dtype)
print("Feature count:", len(features))
display(data[["mean_radius", "mean_area", "target", "diagnosis"]].head())

In [ ]:
# IQR outlier screening. Values are flagged, not automatically deleted.
q1 = data[features].quantile(0.25)
q3 = data[features].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outlier_mask = data[features].lt(lower) | data[features].gt(upper)
outlier_counts = outlier_mask.sum().sort_values(ascending=False)
rows_with_outlier = int(outlier_mask.any(axis=1).sum())

print("Rows with at least one IQR outlier:", rows_with_outlier, "of", len(data))
display(outlier_counts.head(10).to_frame("IQR_outlier_count"))

row_outlier_by_class = (
    pd.Series(outlier_mask.any(axis=1), index=data.index, name="has_outlier")
    .groupby(data["diagnosis"], observed=False).mean() * 100
)
print("Percentage of rows with >=1 outlier, by diagnosis:")
display(row_outlier_by_class.to_frame("percent"))

**Preprocessing decision:** No rows are removed or imputed if the quality checks confirm that no missing or duplicate records exist. Outliers are kept because large or irregular nuclei may be part of the real signal distinguishing the diagnosis groups. Deleting them could remove important malignant observations and bias the analysis. Robust summaries and rank-based statistical tests are therefore used alongside visual inspection.

## 7. Exploratory Data Analysis

### 7.1 Compare key measurements by diagnosis

In [ ]:
key_features = [
    "mean_radius", "mean_texture", "mean_perimeter", "mean_area",
    "mean_smoothness", "mean_compactness", "mean_concavity",
    "mean_concave_points"
]

medians = data.groupby("diagnosis", observed=False)[key_features].median().T
medians["malignant_to_benign_ratio"] = medians["Malignant"] / medians["Benign"]
display(medians)

In [ ]:
# Histograms for selected measurements
plot_features = ["mean_radius", "mean_texture", "mean_area", "mean_concave_points"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, plot_features):
    sns.histplot(data=data, x=col, hue="diagnosis", stat="density",
                 common_norm=False, element="step", fill=False, ax=ax)
    ax.set_title(col.replace("_", " ").title())
plt.tight_layout()
plt.show()

In [ ]:
# Box plots to compare class distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
box_features = [
    "mean_radius", "mean_perimeter", "mean_area",
    "mean_texture", "mean_smoothness", "mean_concave_points"
]
for ax, col in zip(axes.flat, box_features):
    sns.boxplot(data=data, x="diagnosis", y=col,
                order=["Benign", "Malignant"], ax=ax)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 7.2 Feature separation: effect sizes and statistical tests

In [ ]:
def cohens_d(a, b):
    # Difference in means divided by pooled standard deviation
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    pooled = np.sqrt(
        ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
        / (len(a) + len(b) - 2)
    )
    return (a.mean() - b.mean()) / pooled

mal = data[data["diagnosis"] == "Malignant"]
ben = data[data["diagnosis"] == "Benign"]

effect_rows = []
for col in features:
    p_value = stats.mannwhitneyu(mal[col], ben[col], alternative="two-sided").pvalue
    effect_rows.append({
        "feature": col,
        "cohens_d": cohens_d(mal[col], ben[col]),
        "p_value": p_value
    })

effects = pd.DataFrame(effect_rows)
effects["bonferroni_significant"] = effects["p_value"] < (0.05 / len(features))
effects = effects.sort_values("cohens_d", key=lambda s: s.abs(), ascending=False)
display(effects.head(10))
print("Significant after Bonferroni correction:",
      int(effects["bonferroni_significant"].sum()), "of", len(features))
print("Features with large effect (|d| >= 0.8):",
      int((effects["cohens_d"].abs() >= 0.8).sum()))

In [ ]:
top_effects = effects.head(12).sort_values("cohens_d")
plt.figure(figsize=(9, 6))
sns.barplot(data=top_effects, x="cohens_d", y="feature")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Features with the Largest Class Effect Sizes")
plt.xlabel("Cohen's d (malignant minus benign)")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Interpretation:** Cohen's d measures the size of the difference between groups, while the Mann–Whitney U test assesses whether their rank distributions differ. The Bonferroni threshold controls the family-wise error rate across 30 feature tests. Statistical significance alone does not imply practical importance; effect size should also be considered.

### 7.3 Correlation between features

In [ ]:
mean_features = [c for c in features if c.startswith("mean_")]
corr = data[mean_features].corr(method="pearson")

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix: Mean Features")
plt.tight_layout()
plt.show()

# Show strongest unique pairs
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print("Strongest absolute correlations:")
display(corr_pairs.reindex(corr_pairs.abs().sort_values(ascending=False).index).head(10).to_frame("correlation"))

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=data, x="mean_radius", y="mean_concave_points",
                hue="diagnosis", alpha=0.75)
plt.title("Mean Radius vs Mean Concave Points")
plt.tight_layout()
plt.show()

### 7.4 Principal component analysis (PCA)

In [ ]:
# Scale before PCA because measurements use different units and ranges.
X_scaled = StandardScaler().fit_transform(data[features])
pca = PCA(random_state=RANDOM_STATE)
pca.fit(X_scaled)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

pca_summary = pd.DataFrame({
    "component": np.arange(1, len(features) + 1),
    "individual_variance": pca.explained_variance_ratio_,
    "cumulative_variance": cumulative_variance
})
display(pca_summary.head(12))

n90 = int(np.argmax(cumulative_variance >= 0.90) + 1)
n95 = int(np.argmax(cumulative_variance >= 0.95) + 1)
print("Components needed for 90% variance:", n90)
print("Components needed for 95% variance:", n95)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(features) + 1), cumulative_variance, marker="o")
plt.axhline(0.90, linestyle="--", label="90%")
plt.axhline(0.95, linestyle="--", label="95%")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA: Cumulative Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()

### 7.5 Baseline classification check

In [ ]:
# Scaling is inside the pipeline: each fold learns scaling from training data only.
X = data[features]
y = data["target"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

majority_baseline = data["target"].value_counts(normalize=True).max()
print("Fold accuracies:", np.round(scores, 3))
print(f"Mean cross-validated accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

**Model interpretation and caution:** This logistic-regression experiment is a simple separability check, not a clinical model. Accuracy should be read alongside the majority-class baseline and the class imbalance. In a real screening context, sensitivity, specificity, false negatives and external validation would be essential; accuracy alone is not sufficient.

## 8. Key findings and interpretation

The analysis is expected to show that the dataset contains 569 observations and 30 numerical predictor features. The diagnosis classes are moderately imbalanced, with approximately 62.7% benign and 37.3% malignant cases. Initial quality checks indicate no missing values, duplicate rows or negative feature measurements. Zero values in concavity-related fields are meaningful measurements rather than automatically invalid data.

The strongest differences between the diagnosis groups are generally associated with size and boundary irregularity. Features such as worst concave points, worst perimeter, mean concave points and worst radius have very large effect sizes in the supplied analysis. Median area and concavity-related measurements are higher in malignant cases. Texture and smoothness tend to show more overlap, making them weaker standalone separators. The correlation analysis also demonstrates redundancy: radius, perimeter and area are strongly correlated because they describe related geometric properties. PCA provides another view of this redundancy by showing how a smaller number of components can capture most of the standardised feature variance.

Outlier screening identifies a substantial number of extreme values, but these are retained because they may represent genuine biological variation and can be more frequent among malignant observations. A standardised logistic-regression pipeline provides a baseline estimate of class separability using stratified five-fold cross-validation. The result should be compared with the majority-class baseline and must not be interpreted as evidence of clinical readiness.

## 9. Recommendations

- Prioritise concave-points, perimeter, radius and area measurements when exploring patterns or building a baseline model.
- Treat highly correlated measurements carefully. Consider feature selection, regularisation or PCA rather than assuming all 30 variables provide independent information.
- Retain plausible outliers and use robust summaries and rank-based tests when distributions are skewed.
- Report cross-validation results alongside a simple baseline, and keep preprocessing inside the cross-validation pipeline to prevent data leakage.
- For any future clinical research, evaluate sensitivity and false-negative rates, validate on independent populations and involve qualified medical professionals.

## 10. Limitations

- The dataset contains 569 cases and comes from a single institutional source, so results may not generalise to other populations or imaging settings.
- The notebook analyses pre-computed measurements rather than the original images.
- The classes are moderately imbalanced.
- Statistical association and cross-validated accuracy do not establish causation or clinical usefulness.
- This notebook is for education and data-analysis practice only. It is not a diagnostic tool.

## 11. References

1. Wolberg, W. H., Street, W. N. and Mangasarian, O. L. *Breast Cancer Wisconsin (Diagnostic) Dataset*. UCI Machine Learning Repository.
2. scikit-learn documentation: `sklearn.datasets.load_breast_cancer`.
3. pandas, NumPy, SciPy, Matplotlib and Seaborn documentation.

---

**End of Task 3 report.**